In [1]:
import sklearn
import math
import numpy as np
import pandas as pd
import os
import csv
from tqdm import tqdm
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter

#固定随机种子，确保深度学习可重复性

In [2]:
def same_seed(seed):
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

#划分数据集

In [3]:
def train_valid_split(data_set, valid_ratio, seed):
    valid_data_size=int(len(data_set)*valid_ratio)
    train_data_size=len(data_set)-valid_data_size
    train_data,valid_data=random_split(data_set,lengths=[train_data_size,valid_data_size],generator=torch.Generator().manual_seed(seed))
    return np.array(train_data),np.array(valid_data)

#选择特征

In [4]:
def select_feat(train_data,valid_data,test_data,select_all=True,k=15):
    y_train=train_data[:,-1]
    y_valid=valid_data[:,-1]

    raw_x_train=train_data[:,:-1]
    raw_x_valid=valid_data[:,:-1]
    raw_x_test=test_data

    scaler=StandardScaler()
    raw_x_train=scaler.fit_transform(raw_x_train)
    raw_x_valid=scaler.transform(raw_x_valid)
    raw_x_test=scaler.transform(raw_x_test)

    if select_all:
        feat_idx=list(range(raw_x_train.shape[1]))
    else:
      correlations=[]
      for i in range(raw_x_train.shape[1]):
          corr=np.corrcoef(raw_x_train[:,i],y_train)[0,1]
          correlations.append(abs(corr))
      correlations=np.nan_to_num(correlations)
      feat_idx=np.argsort(correlations)[::-1][:k].tolist()
      print(f"[{k} Features selected by Correlation] indices: {feat_idx}")
    return raw_x_train[:,feat_idx], raw_x_valid[:,feat_idx], raw_x_test[:,feat_idx],y_train, y_valid


#数据集

In [5]:
class COVID119Dataset(Dataset):
    def __init__(self,features,targets=None):
        if targets is None:
            self.targets=targets
        else:
             self.targets=torch.FloatTensor(targets)
        self.features=torch.FloatTensor(features)

    def __getitem__(self, idx):
        if self.targets is None:
            return self.features[idx]
        else:
            return self.features[idx],self.targets[idx]

    def __len__(self):
        return len(self.features)


#神经网络

In [6]:
class My_Model(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layers=nn.Sequential(
        nn.Linear(input_dim,out_features=64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(0.15),
        nn.Linear(in_features=64,out_features=16),
        nn.BatchNorm1d(16),
        nn.ReLU(),
        nn.Dropout(0.15),
        nn.Linear(in_features=16,out_features=1)
       )

    def forward(self,x):
        x=self.layers(x)
        x=x.squeeze(1)
        return x


#参数设置

In [7]:
device='cuda' if torch.cuda.is_available()else'cpu'
config={
    'seed':5201314,
    'select_all':True,
    'valid_ratio':0.2,
    'n_epochs':3000,
    'batch_size':256,
    'learning_rate':1e-3,
    'early_stop':400,
    'save_path':'./models/model.ckpt'
}

#训练过程

In [8]:
def trainer(train_loader, valid_loader, model, config, device):
    criterion=nn.MSELoss(reduction='mean')
    optimizer=torch.optim.AdamW(model.parameters(), lr=config['learning_rate'] ,weight_decay=1e-5)
    writer=SummaryWriter()
    if not os.path.isdir('./models'):
        os.mkdir('./models')
    n_epochs=config['n_epochs']
    best_loss=math.inf
    step=0
    early_stop_count=0

    for epoch in range(n_epochs):
        model.train()
        loss_record=[]
        train_pbar=tqdm(train_loader,position=0,leave=False)

        #train loop
        for x, y in train_pbar:
            optimizer.zero_grad()
            x,y=x.to(device),y.to(device)
            pred=model(x)
            loss=criterion(pred,y)
            loss.backward()
            optimizer.step()
            step+=1
            loss_record.append(loss.detach().item())
            #显示训练过程
            train_pbar.set_description(f'Epoch{epoch+1}/{n_epochs}')
            train_pbar.set_postfix({'loss':loss.detach().item()})

        mean_train_loss=sum(loss_record)/len(loss_record)
        writer.add_scalar('Loss/train' ,mean_train_loss,step)

        #valid loop
        model.eval()
        loss_record=[]
        for x,y in valid_loader:
            x,y=x.to(device),y.to(device)
            with torch.no_grad():
                pred=model(x)
                loss=criterion(pred,y)
            loss_record.append(loss.item())

        mean_valid_loss=sum(loss_record)/len(loss_record)
        print(f'Epoch[{epoch+1}/{n_epochs}]: Train loss:{mean_train_loss:.4f},Valid loss:{mean_valid_loss:.4f}')
        writer.add_scalar('Loss/Valid',mean_valid_loss,step)

        if mean_valid_loss<best_loss:
            best_loss=mean_valid_loss
            torch.save(model.state_dict(),config['save_path'])
            print('saving model with loss{:.3f}'.format(best_loss))
            early_stop_count=0
        else:early_stop_count+=1

        if early_stop_count>=config['early_stop']:
            print('\n Model is not improving, so we helt the training session')
            return

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
 #准备工作
same_seed(config['seed'])
train_data=pd.read_csv('/content/drive/MyDrive/covid_train.csv').values
test_data=pd.read_csv('/content/drive/MyDrive/covid_test.csv').values
train_data,valid_data=train_valid_split(train_data,config['valid_ratio'],config['seed'])
print(f'train_data_size:{train_data.shape}, valid_data_size:{valid_data.shape}, test_data_size{test_data.shape}')
x_train,x_valid,x_test,y_train,y_valid=select_feat(train_data,valid_data,test_data)
print(f'the number of features:{x_train.shape[1]}')
train_dataset=COVID119Dataset(x_train,y_train)
valid_dataset=COVID119Dataset(x_valid,y_valid)
test_dataset=COVID119Dataset(x_test)
train_loader=DataLoader(train_dataset,batch_size=config['batch_size'],shuffle=True,pin_memory=True)
valid_loader=DataLoader(valid_dataset,batch_size=config['batch_size'],shuffle=True,pin_memory=True)
test_loader=DataLoader(test_dataset,batch_size=config['batch_size'],shuffle=False,pin_memory=True)

#开始训练
model=My_Model(input_dim=x_train.shape[1]).to(device)
trainer(train_loader,valid_loader,model,config,device)

#预测
def predict(test_loader,model,device):
    model.eval()
    preds=[]
    for x in tqdm(test_loader):
            x=x.to(device)
            with torch.no_grad():
                pred=model(x)
                preds.append(pred.detach().cpu())
    preds=torch.cat(preds,dim=0).numpy()
    return preds

def save_pred(preds,file):
    with open(file,'w') as fp:
                writer=csv.writer(fp)
                writer.writerow(['id','tested_positive'])
                for i,p in enumerate(preds):
                    writer.writerow([i,p])
model=My_Model(input_dim=x_train.shape[1]).to(device)
model.load_state_dict(torch.load(config['save_path']))
preds=predict(test_loader,model,device)
save_pred(preds,'pred.csv')

train_data_size:(2160, 57), valid_data_size:(540, 57), test_data_size(893, 56)
the number of features:56


Epoch[1/3000]: Train loss:312.4197,Valid loss:306.9940
saving model with loss306.994


Epoch[2/3000]: Train loss:303.3821,Valid loss:326.3060


Epoch[3/3000]: Train loss:299.0141,Valid loss:311.0871


Epoch[4/3000]: Train loss:293.3528,Valid loss:302.9320
saving model with loss302.932


...

Epoch[1566/3000]: Train loss:3.7409,Valid loss:1.2597


Epoch[1567/3000]: Train loss:4.0077,Valid loss:1.4062


Epoch[1568/3000]: Train loss:3.9318,Valid loss:1.2132


Epoch[1569/3000]: Train loss:3.9331,Valid loss:1.1009

 Model is not improving, so we helt the training session


100%|██████████| 4/4 [00:00<00:00, 399.80it/s]
